In [1]:
from sage.repl.interface_magic import InterfaceMagic

# 1. Your working interface definition
m2 = Macaulay2(command='/opt/homebrew/bin/M2')

# 2. Get the current Jupyter notebook shell
shell = get_ipython()

# 3. Bind the 'm2' object to a Jupyter magic interface
interface = InterfaceMagic("m2", m2)

# 4. Register %%m2 as a recognized cell magic
shell.register_magic_function(interface.cell_magic_factory(), magic_name="m2", magic_kind='cell')

In [2]:
m2('loadPackage "NormalToricVarieties"')
m2('loadPackage "RankThreeTorics"')
m2('loadPackage "ToricExtras"')
m2('loadPackage "Topcom"')

RankThreeTorics

In [3]:
%%m2
loadedPackages
installedPackages()

{RankThreeTorics, ToricExtras, NormalToricVarieties, Truncations, Polyhedra, Varieties, Isomorphism, Saturation, Elimination, OldChainComplexes, Schubert2, TangentCone, SimpleDoc, ReesAlgebra, PrimaryDecomposition, MinimalPrimes, PackageCitations, OnlineLookup, LLLBases, InverseSystems, IntegralClosure, ConwayPolynomials, Classic, Core}

List

{NormalToricVarieties, Polyhedra, RankThreeTorics, ToricExtras}

List


In [4]:
%%m2

indextovector = (i, X) -> (
    R = rays X;
    return R#i
);

checkvectorincone = (u,c,X) -> (
    coneinvectors = for i in c list indextovector(i, X);
    coneobject = coneFromVData transpose matrix coneinvectors;
    uMatrix = transpose matrix {u};
    return inInterior(uMatrix,coneobject)
);

-- input u = {1,1,1}, output c = {1,2,3}
vectortocone = (u,X) -> (
    for i in (0,1,2,3) do (
        for c in (orbits X)#i do (
            if checkvectorincone(u,c,X) == true then (
                return c;
            )
        )
    )
);

primitivePoints = (X,n) -> (
    pts = {};
    R = rays X;
    for a from -n to n do
      for b from -n to n do
        for c from -n to n do (
          if gcd(a, gcd(b,c)) == 1 and not member({a,b,c}, R) then pts = append(pts, {a,b,c});
        );
    return pts
);

isominlist = (X,L) -> (
    for Y in L do if areIsomorphic(X,Y) == true then return true;
    return false
);

removeisomorphic = L -> (
    newlist = {};
    for X in L do if isominlist(X,newlist) == false then newlist = append(newlist,X);
    return newlist
);

doblowupofvector = (X,v) -> (
    c = vectortocone(v,X);
    return toricBlowup(c,X,v)
);

blowuplist = (X,n) -> (
    points = primitivePoints(X,n);
    outputlist = for p in points list doblowupofvector(X,p);
    return removeisomorphic(outputlist)
);

Picardnumberblowups = (X,r,n) -> (
    counter = 1;
    book = new MutableHashTable from {1=>{X}};
    while counter < r do (
        templist = {};
        for Y in book#counter do (
            Bl = blowuplist (Y,n);
            templist = join(templist, Bl);
        );
    counter = counter + 1;
    book#counter = removeisomorphic(templist);
    );
    return book
);

blowuplistwoiso = (X,n) -> (
    points = primitivePoints(X,n);
    outputlist = for p in points list doblowupofvector(X,p);
    return outputlist
);

Picardnumberblowupswoiso = (X,r,n) -> (
    counter = 1;
    book = new MutableHashTable from {1=>{X}};
    while counter < r do (
        templist = {};
        for Y in book#counter do (
            Bl = blowuplistwoiso (Y,n);
            templist = join(templist, Bl);
        );
    counter = counter + 1;
    book#counter = templist;
    );
    return book
);

blowuplistwoisosmooth = (X,n) -> (
    points = primitivePoints(X,n);
    outputlist = {};
    for p in points do (
        B = doblowupofvector(X,p);
        if isSmooth B then outputlist = append(outputlist, B);
    );
    return outputlist
);

Picardnumberblowupswoisosmooth = (X,r,n) -> (
    counter = 1;
    book = new MutableHashTable from {1=>{X}};
    while counter < r do (
        templist = {};
        for Y in book#counter do (
            Bl = blowuplistwoisosmooth (Y,n);
            templist = join(templist, Bl);
        );
    counter = counter + 1;
    book#counter = templist;
    );
    return book
);

getFVector = X -> apply(dim X + 1, i -> length (orbits X)#i);
                        
fastRemoveIsomorphic = L -> (
    bins = new MutableHashTable;
    
    -- Step 1: Group varieties by their f-vector
    for X in L do (
        inv = getFVector(X);
        if not bins#?inv then bins#inv = {};
        bins#inv = append(bins#inv, X);
    );

    uniqueList = {};
    
    -- Step 2: Check for isomorphisms ONLY within each bin
    for inv in keys bins do (
        bin = bins#inv;
        uniqueInBin = {};
        
        for X in bin do (
            isIso = false;
            for Y in uniqueInBin do (
                if areIsomorphic(X,Y) then (
                    isIso = true;
                    break; -- We found a match, stop checking this bin
                );
            );
            if not isIso then uniqueInBin = append(uniqueInBin, X);
        );
        uniqueList = join(uniqueList, uniqueInBin);
    );
    
    return uniqueList
);
                        
blowuplistwisosmooth = (X,n) -> (
    points = primitivePoints(X,n);
    for p in points do (
        B = doblowupofvector(X,p);
        if isSmooth B then outputlist = append(outputlist, B);
    );
    return fastRemoveIsomorphic(outputlist)
);

Picardnumberblowupswisosmooth = (X,r,n) -> (
    counter = 1;
    book = new MutableHashTable from {1=>{X}};
    while counter < r do (
        templist = {};
        for Y in book#counter do (
            Bl = blowuplistwisosmooth (Y,n);
            templist = join(templist, Bl);
        );
    counter = counter + 1;
    book#counter = templist;
    );
    return book
);

checkIntegerSpansmoothsubv = (w, y) -> (
    -- Convert matrices to flat lists (if they aren't already)
    wList = flatten w;
    yList = flatten y;
    -- Find the index of the first non-zero entry in y
    i = 0;
    while yList#i == 0 do i = i + 1;
    -- In Macaulay2, '%' works perfectly on plain integers!
    -- If the remainder is 0, 'a' is an integer.
    isInteger = (wList#i % yList#i == 0);
    if isInteger then (
        -- '//' also works perfectly on plain integers for exact division
        -- a = wList#i // yList#i; 
        -- print("a is the integer: " | toString a);
        return true;
    ) else (
        -- print("a is a fraction!");
        return false;
    )
);


smoothsubv = (P,X) -> (
    if computel(P,X) == 1 then(
        pvectors = for i in P list indextovector(i,X);
        w = sum pvectors;
        c = vectortocone(w,X);
        y = for j in c list indextovector(j,X);
        return checkIntegerSpansmoothsubv(w,y);
    ) else return false;
);
                                       
                                       -- P = {3,0}
relationssum = (P,X) -> (
    pvectors = for i in P list indextovector(i,X);
    w = sum pvectors;
    if w == {0,0,0} then return pi;
    c = vectortocone(w,X);
    cvectors = for j in c list indextovector(j,X);
    combinedmatrix = join(pvectors,cvectors);
    M = transpose matrix combinedmatrix;
    rltns = ker M;
    G = gens rltns;
    total = sum flatten entries G;
    return total
);

computel = (P,X) -> (
    pvectors = for i in P list indextovector(i,X);
    w = sum pvectors;
    if w == {0,0,0} then return 0;
    c = vectortocone(w,X);
    return length c
);

numberofflops = X -> (
    fcounter = 0;
    PCS = toricPrimitiveCollections(rays X, max X);
    for PC in PCS do (if (length PC == 2 and relationssum(PC,X) == 0) then fcounter=fcounter+1);
    return fcounter
);

numberofblowdownsnotfloips = X -> (
    bcounter = 0;
    PCS = toricPrimitiveCollections(rays X, max X);
    for PC in PCS do if computel(PC,X) == 1 then bcounter=bcounter+1;
    return bcounter
);

numberofflopsandblowdownsnotfloips = X -> (
    fcounter = 0;
    bcounter = 0;
    PCS = toricPrimitiveCollections(rays X, max X);
    for PC in PCS do (if (length PC == 2 and relationssum(PC,X) == 0) then fcounter=fcounter+1);
    for PC in PCS do if (computel(PC,X) == 1 and smoothsubv(PC,X)) then bcounter=bcounter+1;
    return (fcounter, bcounter)
);
                                       
                                       coneExistenceCheck = (S, fan) -> (
for cone in fan do (
if isSubset(S, cone) then (
return true;
);
);
return false;
);

properSubsetCheck = (S, fan) -> (
for ray in S do (
if coneExistenceCheck(S-set{ray}, fan) == false then (
return false;
);
);
return true;
);

isPrimitiveCollection = (P, Var) -> (
    if coneExistenceCheck(P, (orbits Var)#0) then (
        return false;
    ) else (
        return properSubsetCheck(P, (orbits Var)#0);
    );
);
    
supsetsOfPrimColl = (E, B) -> (
return set{for P in E-set{B} when isSubset(B, P) list P};
);
    
primitiveCollectionss = (Var) -> (
n = length rays Var;
primColls = select(subsets(toList(0..n-1)), x -> length x > 1);
for P in subsets(toList(0..n-1), 2) do (
if coneExistenceCheck(P, orbits(Var, 0)) == false then (
primColls = primColls - supsetsOfPrimColl(primColls, P);)
else (
primColls = primColls - set{P};
);
);
for i in toList(3..n) do (
for P in subsets(toList(0..n-1), i) do (
if member(P, primColls) == false then continue;
if isPrimitiveCollection(P, Var) then (
primColls = primColls - supsetsOfPrimColl(primColls, P);
) else (
primColls = primColls - set{P};
);
);
);
return sort primColls;
);
    
countflopsandsmoothblowdowns = X -> (
    fcounter = 0;
    bcounter = 0;
    PCS = primitiveCollectionss(X);
    for PC in PCS do (if (length PC == 2 and relationssum(PC,X) == 0) then fcounter=fcounter+1);
    for PC in PCS do if (computel(PC,X) == 1) then bcounter=bcounter+1;
    return (fcounter, bcounter)
);

In [5]:
%%m2
-- 1. Define the weights for the singular blowup.
-- Using asymmetric, Fibonacci-like weights forces deep, unbalanced singularities
w = {2, 2, 3};

-- 2. Define the rays.
-- Indices 0, 1, 2 are the standard basis. 3 is the P^3 compactification ray. 4 is the weighted ray.
RaysYsing = {{1,0,0}, {0,1,0}, {0,0,1}, {-1,-1,-1}, w};

-- 3. Define the maximal cones.
-- We replace the standard origin chart {0,1,2} with its star subdivision around ray 4.
ConesYsing = {
    {4, 1, 2}, {0, 4, 2}, {0, 1, 4}, -- The three singular cones replacing the origin
    {0, 1, 3}, {0, 2, 3}, {1, 2, 3}  -- The untouched projective boundary
};

-- 4. Create the singular projective toric threefold
Y_sing = normalToricVariety(RaysYsing, ConesYsing);

-- Verify our required properties
print isProjective Y_sing -- Will return true
print isSmooth Y_sing     -- Will return false

true
false


In [6]:
%%m2

X = makeSmooth Y_sing
numRays = length (rays X)
picardNumber = numRays - dim X
countflopsandsmoothblowdowns(X)

X

NormalToricVariety

7

4

(1, 8)

Sequence


In [7]:
%%m2
rays X
max X

{{1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, -1}, {2, 2, 3}, {1, 1, 1}, {1, 1, 2}}

List

{{0, 1, 3}, {0, 1, 5}, {0, 2, 3}, {0, 2, 6}, {0, 4, 5}, {0, 4, 6}, {1, 2, 3}, {1, 2, 6}, {1, 4, 5}, {1, 4, 6}}

List


In [8]:
%%m2
needsPackage "Topcom"
sq = transpose matrix {{-1,-1},{-1,1},{1,-1},{1,1},{0,0},{1,0},{-1,0},{0,1},{0,-1}};

tri = topcomRegularFineTriangulation sq


Topcom

Package

         2       9
Matrix ZZ  <-- ZZ

{{2, 4, 5}, {3, 4, 5}, {0, 4, 6}, {1, 4, 6}, {3, 4, 7}, {1, 4, 7}, {2, 4, 8}, {0, 4, 8}}

List


In [9]:
%%m2

-- 1. Define the rays as the COLUMNS of a matrix.
-- The columns correspond to v0, v1, v2, v4(w), v5, v6
A = matrix {
    {1, 0, 0, 2, 1, 1},
    {0, 1, 0, 2, 1, 1},
    {0, 0, 1, 3, 1, 2}
}

Ts = topcomAllTriangulations(A, Homogenize => false)

-- 3. Output the results
<< "Total regular triangulations found: " << #Ts << endl;

for i from 0 to #Ts - 1 do (
    << "--- Triangulation " << i + 1 << " ---" << endl;
    print Ts_i;)

o1000000061 = | 1 0 0 2 1 1 |
              | 0 1 0 2 1 1 |
              | 0 0 1 3 1 2 |

                       3       6
o1000000061 : Matrix ZZ  <-- ZZ

o1000000062 = {{{0, 1, 2}}, {{0, 1, 4}, {0, 2, 4}, {1, 2, 4}}, {{0, 1, 5}, {0, 2, 5}, {1, 2, 5}}, {{0, 1, 3}, {0, 2, 3}, {1, 2, 3}}, {{0, 1, 4}, {0, 2, 5}, {0, 4, 5}, {1, 2, 5}, {1, 4, 5}}, {{0, 1, 3}, {0, 2, 5}, {0, 3, 5}, {1, 2, 5}, {1, 3, 5}}, {{0, 1, 4}, {0, 2, 3}, {0, 3, 4}, {1, 2, 3}, {1, 3, 4}}, {{0, 1, 4}, {0, 2, 5}, {0, 3, 4}, {0, 3, 5}, {1, 2, 5}, {1, 3, 4}, {1, 3, 5}}}

o1000000062 : List

Total regular triangulations found: 8


--- Triangulation 1 ---
{{0, 1, 2}}
--- Triangulation 2 ---
{{0, 1, 4}, {0, 2, 4}, {1, 2, 4}}
--- Triangulation 3 ---
{{0, 1, 5}, {0, 2, 5}, {1, 2, 5}}
--- Triangulation 4 ---
{{0, 1, 3}, {0, 2, 3}, {1, 2, 3}}
--- Triangulation 5 ---
{{0, 1, 4}, {0, 2, 5}, {0, 4, 5}, {1, 2, 5}, {1, 4, 5}}
--- Triangulation 6 ---
{{0, 1, 3}, {0, 2, 5}, {0, 3, 5}, {1, 2, 5}, {1, 3, 5}}
--- Triangulation 7 ---
{{0, 

In [29]:
%%m2

Z = toricProjectiveSpace 3;

-- 1. Define the complete list of global rays for X
-- Indices: 0, 1, 2 are standard. 3 is boundary. 4, 5, 6 are your internal rays.
RaysX = {{1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, -1}, {2, 2, 3}, {1, 1, 1}, {1, 1, 2}}

-- 2. Define the untouched projective boundary cones of P^3
BoundaryCones = {{0, 1, 3}, {0, 2, 3}, {1, 2, 3}}

-- 3. Create a map from TOPCOM indices to Global indices
-- TOPCOM index 3 -> Global 4 (the weight {2,2,3})
-- TOPCOM index 4 -> Global 5 ({1,1,1})
-- TOPCOM index 5 -> Global 6 ({1,1,2})
idxMap = {0, 1, 2, 4, 5, 6}

-- 4. Loop through the TOPCOM output to build and test each X
for i from 0 to #Ts - 1 do (
    T = Ts_i;
    
    -- Translate TOPCOM cones to Global cones using the index map
    mappedCones = apply(T, c -> apply(c, v -> idxMap_v));
    
    -- Combine the mapped local cones with the fixed boundary cones
    GlobalCones = join(mappedCones, BoundaryCones);

    -- 1. Find which global ray indices are actually used in this triangulation
    activeIndices = sort unique flatten GlobalCones;
    
    -- 2. Create a clean list of only the active rays
    CleanRays = apply(activeIndices, i -> RaysX_i);
    
    -- 3. Create a lookup table to map the old global index to the new clean index
    newIndex = new HashTable from apply(#activeIndices, k -> activeIndices_k => k);
    
    -- 4. Remap the cones to use the clean indices
    CleanCones = apply(GlobalCones, c -> apply(c, v -> newIndex_v));
    
    -- 5. Construct the clean candidate toric variety
    X = normalToricVariety(CleanRays, CleanCones);
    
    -- Construct the candidate toric variety
    X = normalToricVariety(RaysX, GlobalCones);
    
    -- Output the results
    << "--- Candidate X from Triangulation " << i + 1 << " ---" << endl;
    << "Is Projective? " << isProjective X << endl;
    << "Is Smooth? " << isSmooth X << endl;
    
    -- If it is smooth, we can analyze it further
    if isSmooth X then (
        -- This is where you would call your custom flop/blowdown check functions
        << " >> This is a valid smooth candidate for the Conjecture test!" << endl;
        print rays X;
        print max X;
        print areIsomorphic(X,Z);
        print countflopsandsmoothblowdowns(X)
    );
    << endl;
)

oo1000000176 = {{1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, -1}, {2, 2, 3}, {1, 1, 1}, {1, 1, 2}}

oo1000000176 : List

oo1000000177 = {{0, 1, 3}, {0, 2, 3}, {1, 2, 3}}

oo1000000177 : List

oo1000000178 = {0, 1, 2, 4, 5, 6}

oo1000000178 : List
/private/var/folders/b_/ldsp3k8d5y7_2db8nv_3fh0m0000gn/T/tmpxa5pyl_n.input:36:63:(3):[6]: error: no method for binary operator _ applied to objects:
            HashTable{0 => 0} (of class HashTable)
                      1 => 1
                      2 => 2
                      3 => 3
      _     0 (of class ZZ)
/private/var/folders/b_/ldsp3k8d5y7_2db8nv_3fh0m0000gn/T/tmpxa5pyl_n.input:36:46:(3):[5]: --back trace--
/private/var/folders/b_/ldsp3k8d5y7_2db8nv_3fh0m0000gn/T/tmpxa5pyl_n.input:36:22:(3):[4]: --back trace--


In [30]:
%%m2

-- Assuming 'Ts' is your list of 8 triangulations from topcomAllTriangulations

-- 1. Define the complete list of global rays for X
RaysX = {{1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, -1}, {2, 2, 3}, {1, 1, 1}, {1, 1, 2}}

-- 2. Define the untouched projective boundary cones of P^3
BoundaryCones = {{0, 1, 3}, {0, 2, 3}, {1, 2, 3}}

-- 3. Map local TOPCOM indices to Global indices for the origin chart
idxMap = {0, 1, 2, 4, 5, 6}

-- 4. Loop through the TOPCOM output to build and test each X
for i from 0 to #Ts - 1 do (
    T = Ts_i;
    
    -- Translate TOPCOM cones to Global cones
    mappedCones = apply(T, c -> apply(c, v -> idxMap_v));
    
    -- Combine with the fixed projective boundary
    GlobalCones = join(mappedCones, BoundaryCones);
    
    -- --- THE PRUNING STEP ---
    -- Find which global ray indices are actually active in this triangulation
    activeIndices = sort unique flatten GlobalCones;
    
    -- Create a clean list of only the active rays
    CleanRays = apply(activeIndices, idx -> RaysX_idx);
    
    -- Create a HashTable to map the old global index to the new clean index
    newIndex = new HashTable from apply(#activeIndices, k -> activeIndices_k => k);
    
    -- Remap the cones to use the clean indices (Using '#' for HashTable lookup!)
    CleanCones = apply(GlobalCones, c -> apply(c, v -> newIndex#v));
    
    -- Construct the clean candidate toric variety
    X = normalToricVariety(CleanRays, CleanCones);
    
    -- --- OUTPUT & CHECKS ---
    << "--- Candidate X from Triangulation " << i + 1 << " ---" << endl;
    << "Is Smooth? " << isSmooth X << endl;
    
    if isSmooth X then (
        -- Check if it is isomorphic to P^3 (Picard number 1)
        if #CleanRays == 4 then (
            << " >> Result: This is just isomorphic to P^3. Skipping." << endl;
        ) else (
            << " >> Result: Valid smooth refinement! Ready for flop/blowdown check." << endl;
            << " >> Picard Number: " << (#CleanRays - 3) << endl;
            -- Call your flop/blowdown counting functions here!
            print rays X;
            print max X;
            print areIsomorphic(X,Z);
            print countflopsandsmoothblowdowns(X)
        );
    );
    << endl;
)

oo1000000183 = {{1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, -1}, {2, 2, 3}, {1, 1, 1}, {1, 1, 2}}

oo1000000183 : List

oo1000000184 = {{0, 1, 3}, {0, 2, 3}, {1, 2, 3}}

oo1000000184 : List

oo1000000185 = {0, 1, 2, 4, 5, 6}

oo1000000185 : List
--- Candidate X from Triangulation 1 ---
Is Smooth? true
 >> Result: This is just isomorphic to P^3. Skipping.

--- Candidate X from Triangulation 2 ---
Is Smooth? true
 >> Result: Valid smooth refinement! Ready for flop/blowdown check.
 >> Picard Number: 2
{{1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, -1}, {1, 1, 1}}
{{0, 1, 3}, {0, 1, 4}, {0, 2, 3}, {0, 2, 4}, {1, 2, 3}, {1, 2, 4}}
false
(0, 1)

--- Candidate X from Triangulation 3 ---
Is Smooth? false

--- Candidate X from Triangulation 4 ---
Is Smooth? false

--- Candidate X from Triangulation 5 ---
Is Smooth? true
 >> Result: Valid smooth refinement! Ready for flop/blowdown check.
 >> Picard Number: 3
{{1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, -1}, {1, 1, 1}, {1, 1, 2}}
{{0, 1, 3}, {0, 1, 4}, {

In [31]:
%%m2

-- Assuming 'Ts' is your list of 8 triangulations from topcomAllTriangulations

-- 1. Define the complete list of global rays for X
RaysX = {{1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, -1}, {3, 5, 8}, {1, 1, 1}, {1, 1, 2}}

-- 2. Define the untouched projective boundary cones of P^3
BoundaryCones = {{0, 1, 3}, {0, 2, 3}, {1, 2, 3}}

-- 3. Map local TOPCOM indices to Global indices for the origin chart
idxMap = {0, 1, 2, 4, 5, 6}

-- 4. Loop through the TOPCOM output to build and test each X
for i from 0 to #Ts - 1 do (
    T = Ts_i;
    
    -- Translate TOPCOM cones to Global cones
    mappedCones = apply(T, c -> apply(c, v -> idxMap_v));
    
    -- Combine with the fixed projective boundary
    GlobalCones = join(mappedCones, BoundaryCones);
    
    -- --- THE PRUNING STEP ---
    -- Find which global ray indices are actually active in this triangulation
    activeIndices = sort unique flatten GlobalCones;
    
    -- Create a clean list of only the active rays
    CleanRays = apply(activeIndices, idx -> RaysX_idx);
    
    -- Create a HashTable to map the old global index to the new clean index
    newIndex = new HashTable from apply(#activeIndices, k -> activeIndices_k => k);
    
    -- Remap the cones to use the clean indices (Using '#' for HashTable lookup!)
    CleanCones = apply(GlobalCones, c -> apply(c, v -> newIndex#v));
    
    -- Construct the clean candidate toric variety
    X = normalToricVariety(CleanRays, CleanCones);
    
    -- --- OUTPUT & CHECKS ---
    << "--- Candidate X from Triangulation " << i + 1 << " ---" << endl;
    << "Is Smooth? " << isSmooth X << endl;
    
    if isSmooth X then (
        -- Check if it is isomorphic to P^3 (Picard number 1)
        if #CleanRays == 4 then (
            << " >> Result: This is just isomorphic to P^3. Skipping." << endl;
        ) else (
            << " >> Result: Valid smooth refinement! Ready for flop/blowdown check." << endl;
            << " >> Picard Number: " << (#CleanRays - 3) << endl;
            -- Call your flop/blowdown counting functions here!
            print rays X;
            print max X;
            print areIsomorphic(X,Z);
            print countflopsandsmoothblowdowns(X)
        );
    );
    << endl;
)

oo1000000191 = {{1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, -1}, {3, 5, 8}, {1, 1, 1}, {1, 1, 2}}

oo1000000191 : List

oo1000000192 = {{0, 1, 3}, {0, 2, 3}, {1, 2, 3}}

oo1000000192 : List

oo1000000193 = {0, 1, 2, 4, 5, 6}

oo1000000193 : List
--- Candidate X from Triangulation 1 ---
Is Smooth? true
 >> Result: This is just isomorphic to P^3. Skipping.

--- Candidate X from Triangulation 2 ---
Is Smooth? true
 >> Result: Valid smooth refinement! Ready for flop/blowdown check.
 >> Picard Number: 2
{{1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, -1}, {1, 1, 1}}
{{0, 1, 3}, {0, 1, 4}, {0, 2, 3}, {0, 2, 4}, {1, 2, 3}, {1, 2, 4}}
false
(0, 1)

--- Candidate X from Triangulation 3 ---
Is Smooth? false

--- Candidate X from Triangulation 4 ---
Is Smooth? false

--- Candidate X from Triangulation 5 ---
Is Smooth? true
 >> Result: Valid smooth refinement! Ready for flop/blowdown check.
 >> Picard Number: 3
{{1, 0, 0}, {0, 1, 0}, {0, 0, 1}, {-1, -1, -1}, {1, 1, 1}, {1, 1, 2}}
{{0, 1, 3}, {0, 1, 4}, {

In [18]:
%%m2

needsPackage "Topcom"
needsPackage "NormalToricVarieties"

Z = toricProjectiveSpace 3;

-- 1. Define your weight (You can change this to any valid weight later!)
w = {3, 4, 7};

-- 2. Automatically generate internal primitive lattice points
-- This uses ceiling integer division to find points inside the fundamental parallelepipeds
pts1 = apply(1 .. w_2 - 1, k -> { (w_0 * k + w_2 - 1) // w_2, (w_1 * k + w_2 - 1) // w_2, k });
pts2 = apply(1 .. w_1 - 1, k -> { (w_0 * k + w_1 - 1) // w_1, k, (w_2 * k + w_1 - 1) // w_1 });
pts3 = apply(1 .. w_0 - 1, k -> { k, (w_1 * k + w_0 - 1) // w_0, (w_2 * k + w_0 - 1) // w_0 });

-- Combine, remove duplicates, and filter for primitive vectors
allPts = unique join(pts1, pts2, pts3);
gcd3 = v -> gcd(v_0, gcd(v_1, v_2));
InternalRays = select(allPts, v -> gcd3(v) == 1);

<< "Found " << #InternalRays << " internal primitive rays." << endl;

-- 3. Set up the rays for TOPCOM and the Global Projective Variety
OriginRays = {{1,0,0}, {0,1,0}, {0,0,1}, w};
LocalRays = join(OriginRays, InternalRays);

BoundaryRay = {-1, -1, -1};
RaysX = join({{1,0,0}, {0,1,0}, {0,0,1}, BoundaryRay, w}, InternalRays);

-- 4. Build the matrix and run TOPCOM
A = matrix apply(3, i -> apply(LocalRays, v -> v_i));

<< "Running TOPCOM on " << #LocalRays << " total rays..." << endl;
Ts = topcomAllTriangulations(A, Homogenize => false);
<< "Total regular triangulations found: " << #Ts << endl << endl;

-- 5. Prepare the fixed boundaries and dynamic index map
BoundaryCones = {{0, 1, 3}, {0, 2, 3}, {1, 2, 3}};
-- Maps local TOPCOM indices to global RaysX indices
idxMap = join({0, 1, 2, 4}, apply(#InternalRays, i -> i + 5));

-- 6. Evaluate all triangulations
for i from 0 to #Ts - 1 do (
    T = Ts_i;
    
    -- Map to global and combine with boundary
    mappedCones = apply(T, c -> apply(c, v -> idxMap_v));
    GlobalCones = join(mappedCones, BoundaryCones);
    
    -- --- THE PRUNING STEP ---
    activeIndices = sort unique flatten GlobalCones;
    CleanRays = apply(activeIndices, idx -> RaysX_idx);
    
    newIndex = new HashTable from apply(#activeIndices, k -> activeIndices_k => k);
    CleanCones = apply(GlobalCones, c -> apply(c, v -> newIndex#v));
    
    -- Construct the candidate
    X = normalToricVariety(CleanRays, CleanCones);
    
    -- --- OUTPUT & CHECKS ---
    if isSmooth X then (
        -- Silently skip the trivial P^3 case
        if #CleanRays != 4 then (
            << "--- Smooth Candidate X from Triangulation " << i + 1 << " ---" << endl;
            << " >> Picard Number: " << (#CleanRays - 3) << endl;
            
            -- Call your custom function and extract the sequence
            counts = countflopsandsmoothblowdowns(X);
            numFlops = counts#0;
            numBlowdowns = counts#1;
            
            << " >> Flops: " << numFlops << ", Smooth Blowdowns: " << numBlowdowns << endl;
            
            -- THE COUNTEREXAMPLE CHECK
            if numFlops == 0 and numBlowdowns == 0 then (
                << endl;
                << "==========================================" << endl;
                << "   COUNTEREXAMPLE FOUND! " << endl;
                << "   Triangulation index: " << i + 1 << endl;
                << "==========================================" << endl;
                
                -- Print the defining data so you can save it
                << "Rays of X: " << CleanRays << endl;
                << "Maximal Cones of X: " << CleanCones << endl;
                
                -- Halt the loop completely
                break;
            );
            << endl;
        );
    );
)

Interrupting Macaulay2...


KeyboardInterrupt: Ctrl-c pressed while running Macaulay2

In [23]:
%%m2

Z = toricProjectiveSpace 3;

needsPackage "Topcom"
needsPackage "NormalToricVarieties"

-- Make sure your countflopsandsmoothblowdowns(X) function is loaded before running this!

huntForCounterexample = (weightsList) -> (
    foundCounterexample = false;
    
    for w in weightsList do (
        << "==========================================" << endl;
        << "STARTING SEARCH FOR WEIGHT w = " << w << endl;
        << "==========================================" << endl;

        -- 1. Automatically generate internal primitive lattice points
        pts1 = apply(1 .. w_2 - 1, k -> { (w_0 * k + w_2 - 1) // w_2, (w_1 * k + w_2 - 1) // w_2, k });
        pts2 = apply(1 .. w_1 - 1, k -> { (w_0 * k + w_1 - 1) // w_1, k, (w_2 * k + w_1 - 1) // w_1 });
        pts3 = apply(1 .. w_0 - 1, k -> { k, (w_1 * k + w_0 - 1) // w_0, (w_2 * k + w_0 - 1) // w_0 });

        allPts = unique join(pts1, pts2, pts3);
        gcd3 = v -> gcd(v_0, gcd(v_1, v_2));
        InternalRays = select(allPts, v -> gcd3(v) == 1);

        << " >> Found " << #InternalRays << " internal primitive rays." << endl;

        -- 2. Set up the rays for TOPCOM
        OriginRays = {{1,0,0}, {0,1,0}, {0,0,1}, w};
        LocalRays = join(OriginRays, InternalRays);

        BoundaryRay = {-1, -1, -1};
        RaysX = join({{1,0,0}, {0,1,0}, {0,0,1}, BoundaryRay, w}, InternalRays);

        A = matrix apply(3, i -> apply(LocalRays, v -> v_i));

        << " >> Running TOPCOM on " << #LocalRays << " total rays..." << endl;
        Ts = topcomAllTriangulations(A, Homogenize => false);
        << " >> Total regular triangulations found: " << #Ts << endl << endl;

        -- 3. Prepare boundaries and index maps
        BoundaryCones = {{0, 1, 3}, {0, 2, 3}, {1, 2, 3}};
        idxMap = join({0, 1, 2, 4}, apply(#InternalRays, i -> i + 5));

        -- 4. Evaluate all triangulations for this specific weight
        for i from 0 to #Ts - 1 do (
            T = Ts_i;
            
            mappedCones = apply(T, c -> apply(c, v -> idxMap_v));
            GlobalCones = join(mappedCones, BoundaryCones);
            
            activeIndices = sort unique flatten GlobalCones;
            CleanRays = apply(activeIndices, idx -> RaysX_idx);
            
            newIndex = new HashTable from apply(#activeIndices, k -> activeIndices_k => k);
            CleanCones = apply(GlobalCones, c -> apply(c, v -> newIndex#v));
            
            X = normalToricVariety(CleanRays, CleanCones);
            
            if isSmooth X then (
                if #CleanRays != 4 then (
                    
                    counts = countflopsandsmoothblowdowns(X);
                    numFlops = counts#0;
                    numBlowdowns = counts#1;
                    
                    if numFlops == 0 and numBlowdowns == 0 then (
                        << endl;
                        << "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!" << endl;
                        << "   COUNTEREXAMPLE FOUND FOR w = " << w << "  " << endl;
                        << "   Triangulation index: " << i + 1 << endl;
                        << "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!" << endl;
                        << "Rays of X: " << CleanRays << endl;
                        << "Maximal Cones of X: " << CleanCones << endl;
                        
                        foundCounterexample = true;
                        break; -- Break out of the triangulation loop
                    );
                );
            );
        );
        
        if foundCounterexample then break; -- Break out of the weight loop
        << "Finished checking w = " << w << ". No counterexample found." << endl << endl;
    );
    
    if not foundCounterexample then (
        << "==========================================" << endl;
        << "SEARCH COMPLETE. No counterexamples found in the provided list." << endl;
        << "==========================================" << endl;
    );
)

-- Define your list of target weights to test
myWeights = {{3, 5, 8}}

-- Execute the batch hunt
huntForCounterexample(myWeights)

Interrupting Macaulay2...


KeyboardInterrupt: Ctrl-c pressed while running Macaulay2